## Problem Statement
- **Role**: Create and host a website that is able to provide the comparison of prices between top 5 available online e-commerce websites the item's price should be mentioned along with the website name where its available.

- **Task**: Prepare for the upcoming festive season by analyzing competitors' price for given item(s).

- **Objective**: Design a competitive strategy to either match or surpass the competition's offers.

- **Solution**: Develop an **Agentic System** with the following features:
  - **Dynamic Coding**: Ability to adapt to changes in website structure.
  - **Web Scraping**: Targeted extraction of discounts and promotional offers from competitor platforms.
  - **Data Summarization**: Process and organize extracted information for actionable insights.

- **Outcome**: Enable data-driven decision-making to ensure a competitive edge during the festive season.






In [46]:
import os
os.getcwd()

'd:\\GenAI\\20th_Nov_2025-20251121T144448Z-1-001'

In [47]:
import autogen
import os
from IPython import get_ipython
from autogen import ConversableAgent, AssistantAgent, UserProxyAgent
from autogen.coding import LocalCommandLineCodeExecutor
from IPython.display import HTML, display

In [48]:
from getpass import getpass
OPENAI_KEY = getpass("Enter your api key here : ")

In [57]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_KEY

# Local Code Execution

In [66]:
# Agent1
user_proxy = UserProxyAgent(
    name="User",
    llm_config=False,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"],
    human_input_mode="NEVER",
    code_execution_config=False
)

In [59]:
# Agent2 - Developer/Software/PythonDeveloper
code_writer_agent = ConversableAgent(
    name="CodeWriter",
    system_message="""You are a full stack developer having indepth knowledge of Python development along with angular and react .
    You use your coding skill to solve problems.Follow the requirements given by the user . First understand the requirements and then prepare a plan to solve it.
    Once the task is done, returns 'TERMINATE'.""",
    llm_config={"config_list": [{"model": "gpt-4o-mini"}]},
    
    
)

In [60]:
# Agent3 - Reviewer / Quality Checker
code_reviewer_agent = ConversableAgent(
    name="CodeReviewer",
    system_message="""You are an experienced full stack senior developer acting as a lead reviewer.
    Review the code given to you based on the requirements.Try to check by running the code locally and check the output if its matches with the requirements.
    Once the task is done, returns 'TERMINATE'.""",
    llm_config={"config_list": [{"model": "gpt-4o-mini"}]},
)

In [61]:
local_executor = LocalCommandLineCodeExecutor(
    timeout=30,
    work_dir='./code files')

In [62]:
# Agent4
local_executor_agent = ConversableAgent(
    "local_executor_agent",
    llm_config=False, 
    code_execution_config={"executor": local_executor},
    human_input_mode="NEVER",
)

In [63]:
messages = ["""List down the prices of a given item(s) on the top 5 e-commerce website in India if that item is available with item's name as an input on real time to get idea of the competitior pricing , and thus display the prices along with the website name in the output.
            follow these steps:
                1. a process to take the item name as an input from the user.
                2. a process to search for the item on top 5 e-commerce Indian websites.
                3. a process to get the price of the item along with the website name.
                4. display the output in a user friendly manner.
               """ ,
    """The code is given by an LLM. Review the code based on the above requirements and provide it for execution"""]

In [ ]:
chat_results = autogen.initiate_chats(
    [
        {
            "sender": user_proxy,
            "recipient": code_writer_agent,
            "message": messages[0],
            "summary_method": "last_msg",
        },
        {
            "sender": local_executor_agent,
            "recipient": code_reviewer_agent,
            "message": messages[0] + messages[1],
            "summary_method": "last_msg",
        }
    ]
)

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
def fetch_flipkart_content():
    # Step 1: Download the HTML page
    url = "https://www.flipkart.com/"
    try:
        response = requests.get(url)
        # Check if the request was successful
        if response.status_code == 200:
            html_content = response.text
            
            # Step 2: Remove CSS, JavaScript, and image content
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # Remove script, style and img elements
            for tag in soup(["script", "style", "img"]):
                tag.decompose()
            
            # Get text content
            text_content = soup.get_text(separator="\n")
            
            # Step 3: Save the remaining HTML content
            with open("flipkart_text_content.html", "w", encoding='utf-8') as file:
                file.write(text_content)

            print("HTML content saved successfully.")
        else:
            print(f"Failed to retrieve the webpage. Status code: {response.status_code}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")

In [ ]:
# Call the function to execute
fetch_flipkart_content()

In [64]:
os.listdir('./code files/')

['tmp_code_15d033672deefe5b400bd969a8079c86.bash',
 'tmp_code_161191dd0ea87ebfbf67362a7d2082d1.py',
 'tmp_code_a43ea48a2b364af6a5ded12ec75031f6.py']

In [ ]:
chat_results

In [67]:
# Testing harness: explicit two-step hand-off (CodeWriter -> Executor -> Reviewer)
print('Running explicit two-step hand-off test...')

# Step 1: ask CodeWriter to generate code and capture its result
first_step = {
    "sender": user_proxy,
    "recipient": code_writer_agent,
    "message": messages[0],
    "summary_method": "last_msg",
}
first_chat_results = autogen.initiate_chats([first_step])
first_result = first_chat_results[0]
print('First chat result (CodeWriter) - raw:')
print(first_result)

# Robust extraction helper: try many common shapes returned by autogen/LLM wrappers
def extract_text_from_result(res):
    try:
        # If dict-like
        if isinstance(res, dict):
            # common top-level keys
            for k in ('response','output','answer','text','content'):
                v = res.get(k)
                if isinstance(v, str) and v.strip():
                    return v
                if isinstance(v, dict):
                    for kk in ('content','text'):
                        if kk in v and isinstance(v[kk], str) and v[kk].strip():
                            return v[kk]
                    if 'choices' in v and v['choices']:
                        c = v['choices'][0]
                        if isinstance(c, dict):
                            msg = c.get('message') or c.get('delta') or c.get('output')
                            if isinstance(msg, dict):
                                cont = msg.get('content') or msg.get('text')
                                if cont: return cont
                            if 'text' in c and isinstance(c['text'], str):
                                return c['text']
            # messages list
            msgs = res.get('messages') or res.get('message') or []
            if isinstance(msgs, list):
                for m in reversed(msgs):
                    if isinstance(m, dict):
                        cont = m.get('content') or m.get('text')
                        if cont: return cont
        else:
            # object-like: try attributes
            for attr in ('response','output','result','answer','content','text'):
                v = getattr(res, attr, None)
                if isinstance(v, str) and v.strip():
                    return v
                if isinstance(v, dict):
                    for kk in ('content','text'):
                        if kk in v and isinstance(v[kk], str) and v[kk].strip():
                            return v[kk]
                    if 'choices' in v and v['choices']:
                        c = v['choices'][0]
                        if isinstance(c, dict):
                            msg = c.get('message') or c.get('delta') or c.get('output')
                            if isinstance(msg, dict):
                                cont = msg.get('content') or msg.get('text')
                                if cont: return cont
                            if 'text' in c and isinstance(c['text'], str):
                                return c['text']
            msgs = getattr(res, 'messages', None)
            if msgs:
                for m in reversed(msgs):
                    if isinstance(m, dict):
                        cont = m.get('content') or m.get('text')
                        if cont: return cont
                    else:
                        cont = getattr(m, 'content', None) or getattr(m, 'text', None)
                        if isinstance(cont, str) and cont.strip(): return cont
        # Fallback to stringifying the whole object
        return str(res)
    except Exception as e:
        return str(res)

code_from_writer = extract_text_from_result(first_result) or ''

# Trim common assistant prefixes if present
if code_from_writer and code_from_writer.startswith('Assistant:'):
    code_from_writer = code_from_writer.split(':',1)[1].lstrip()

print('Extracted code (truncated to 1000 chars):')
print((code_from_writer or '')[:1000])

if not code_from_writer or code_from_writer.strip() == '':
    print('No code extracted from CodeWriter result. Here is the full first_result for debugging:')
    print(first_result)
else:
    exec_out = ''
    try:
        if hasattr(local_executor, 'run'):
            exec_out = local_executor.run(code_from_writer)
        elif hasattr(local_executor, 'execute'):
            exec_out = local_executor.execute(code_from_writer)
        else:
            raise AttributeError('Local executor has no run/execute method')
    except Exception as e:
        print('Executor.run/execute failed or not available:', e)
        import subprocess, sys
        work_dir = getattr(local_executor, 'work_dir', './code files')
        temp_path = os.path.join(work_dir, 'temp_code.py')
        os.makedirs(work_dir, exist_ok=True)
        with open(temp_path, 'w', encoding='utf-8') as f:
            f.write(code_from_writer)
        proc = subprocess.run([sys.executable, temp_path], capture_output=True, text=True, cwd=work_dir)
        exec_out = proc.stdout + "\n\n[stderr]\n" + proc.stderr

    print('Execution output (truncated to 2000 chars):')
    print(str(exec_out)[:2000])

    reviewer_message = 'Here is the executed code and its output:\n\n' + (str(exec_out) or 'No output')
    second_step = {
        "sender": local_executor_agent,
        "recipient": code_reviewer_agent,
        "message": reviewer_message,
        "summary_method": "last_msg",
    }
    second_chat_results = autogen.initiate_chats([second_step])
    print('Second chat result (Executor -> Reviewer) - raw:')
    print(second_chat_results[0])



Running explicit two-step hand-off test...

********************************************************************************


Starting a new chat....

********************************************************************************
User (to CodeWriter):

List down the prices of a given item(s) on the top 5 e-commerce website in India if that item is available with item's name as an input on real time to get idea of the competitior pricing , and thus display the prices along with the website name in the output.
            follow these steps:
                1. a process to take the item name as an input from the user.
                2. a process to search for the item on top 5 e-commerce Indian websites.
                3. a process to get the price of the item along with the website name.
                4. display the output in a user friendly manner.
               

********************************************************************************
User (to CodeWriter):

List down the prices of a given item(s) on the top 5 e-commerce website in India if that item is available with item's name as an input on